# [5.1] Gemma from Scratch - Exercises

You will build a tiny Gemma-style decoder from first principles: RMSNorm, RoPE,
grouped-query attention, SwiGLU, a KV cache, and a final parity check against
Hugging Face's `GemmaForCausalLM` reference architecture.

<img src="../../instructions/assets/gemma_decoder_block.svg" width="760">

By the end, you should understand why a modern decoder implementation is not
finished just because it returns logits. The local implementation must match an
independent reference on logits, cache behavior, and deterministic generation.


In [ ]:
import math
import sys
from pathlib import Path

import torch as t
import torch.nn as nn
import torch.nn.functional as F

chapter = "chapter5_modern_architectures"
section = "part1_gemma_from_scratch"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part1_gemma_from_scratch.tests as tests
import part1_gemma_from_scratch.utils as utils

from arena_ext import estimate_inference_memory
from arena_ext.gemma import GemmaCausalLMOutput, GemmaConfig, cache_parity_report

PastKeyValue = tuple[t.Tensor, t.Tensor]


## Exercise 1 - Gemma RMSNorm

Implement Gemma-style RMSNorm. Do not subtract the mean. Gemma stores the learned scale as an offset, so the final multiplier is `1 + weight`.


<details><summary>Expected output</summary>

```text
All tests in `test_gemma_rms_norm` passed!
```

</details>

<details><summary>Help - I implemented LayerNorm and parity fails</summary>

LayerNorm subtracts the mean; RMSNorm does not. Gemma also stores the learned
scale as an offset, so the multiplier is `1 + weight`. A LayerNorm-shaped answer
can pass shape checks and still break reference-logit parity.

</details>


In [ ]:
class GemmaRMSNorm(nn.Module):
    def __init__(self, hidden_size: int, eps: float = 1e-6):
        super().__init__()
        self.weight = nn.Parameter(t.zeros(hidden_size))
        self.eps = eps

    def forward(self, x: t.Tensor) -> t.Tensor:
        raise NotImplementedError()


tests.test_gemma_rms_norm(GemmaRMSNorm)


## Exercise 2 - RoPE

Implement interleaved rotary embeddings. Position zero should be the identity rotation, and every rotation should preserve per-token vector norms.


<img src="../../instructions/assets/rope_even_odd_rotation.svg" width="560">

<details><summary>Expected output</summary>

```text
All tests in `test_rotate_half_interleaved` passed!
All tests in `test_build_rope_cache` passed!
All tests in `test_apply_rope` passed!
```

</details>

<details><summary>Help - my RoPE shapes broadcast incorrectly</summary>

The activation is `[batch, heads, seq, head_dim]`. Indexing the cos/sin cache by
`position_ids` gives `[batch, seq, head_dim]`, then `unsqueeze(1)` restores the
head axis. If you unsqueeze the wrong axis, broadcasting can silently rotate the
wrong positions.

</details>


In [ ]:
def rotate_half_interleaved(x: t.Tensor) -> t.Tensor:
    raise NotImplementedError()


def build_rope_cache(
    seq_len: int,
    head_dim: int,
    *,
    base: float,
    device: t.device,
    dtype: t.dtype,
) -> tuple[t.Tensor, t.Tensor]:
    raise NotImplementedError()


def apply_rope(x: t.Tensor, cos: t.Tensor, sin: t.Tensor, position_ids: t.Tensor) -> t.Tensor:
    raise NotImplementedError()


tests.test_rotate_half_interleaved(rotate_half_interleaved)
tests.test_build_rope_cache(build_rope_cache)
tests.test_apply_rope(apply_rope, build_rope_cache)


## Exercise 3 - Grouped-query helpers and cache-aware masks

Implement contiguous key/value head repetition and a causal mask that uses absolute query positions when a KV cache is present.


<img src="../../instructions/assets/gqa_head_repetition.svg" width="620">

<details><summary>Expected output</summary>

```text
All tests in `test_repeat_kv` passed!
All tests in `test_sliding_window_mask` passed!
```

</details>

<details><summary>Help - my cache parity fails only after the first token</summary>

During generation, a cached query after a prompt of length `n` is at absolute
position `n`, not position zero. Build masks and RoPE `position_ids` from the
absolute positions, then append new key/value states after the old cache.

</details>


In [ ]:
def repeat_kv(hidden_states: t.Tensor, repeats: int) -> t.Tensor:
    raise NotImplementedError()


def build_causal_attention_mask(
    *,
    query_length: int,
    key_length: int,
    past_length: int,
    sliding_window: int | None,
    device: t.device,
) -> t.Tensor:
    raise NotImplementedError()


tests.test_repeat_kv(repeat_kv)
tests.test_sliding_window_mask(build_causal_attention_mask)


## Exercise 4 - SwiGLU MLP

Implement Gemma's gated MLP. The gate projection is passed through SiLU, multiplied elementwise with the up projection, then projected back to the residual stream.


In [ ]:
class GemmaMLP(nn.Module):
    def __init__(self, config: GemmaConfig):
        super().__init__()
        self.gate_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.up_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.down_proj = nn.Linear(config.intermediate_size, config.hidden_size, bias=False)

    def forward(self, x: t.Tensor) -> t.Tensor:
        raise NotImplementedError()


tests.test_gemma_mlp_matches_swiglu_formula(GemmaMLP)


## Exercise 5 - Grouped-query attention with a KV cache

Build the attention block from the helpers above. The test checks the shape of the output, the layout of the key/value cache, and that a one-token cached step appends to the existing cache.


<img src="../../instructions/assets/kv_cache_flow.svg" width="700">

<details><summary>Expected output</summary>

```text
All tests in `test_gemma_attention_shapes_and_cache` passed!
```

</details>


In [ ]:
class GemmaAttention(nn.Module):
    def __init__(self, config: GemmaConfig, layer_idx: int):
        super().__init__()
        self.config = config
        self.layer_idx = layer_idx
        self.num_heads = config.num_attention_heads
        self.num_key_value_heads = config.num_key_value_heads
        self.head_dim = config.head_dim
        self.num_key_value_groups = self.num_heads // self.num_key_value_heads

        self.q_proj = nn.Linear(
            config.hidden_size,
            self.num_heads * self.head_dim,
            bias=config.attention_bias,
        )
        self.k_proj = nn.Linear(
            config.hidden_size,
            self.num_key_value_heads * self.head_dim,
            bias=config.attention_bias,
        )
        self.v_proj = nn.Linear(
            config.hidden_size,
            self.num_key_value_heads * self.head_dim,
            bias=config.attention_bias,
        )
        self.o_proj = nn.Linear(
            self.num_heads * self.head_dim,
            config.hidden_size,
            bias=config.attention_bias,
        )


    def _shape(self, x: t.Tensor, heads: int) -> t.Tensor:
        raise NotImplementedError()

    def forward(
        self,
        hidden_states: t.Tensor,
        *,
        position_ids: t.Tensor,
        cos: t.Tensor,
        sin: t.Tensor,
        attention_mask: t.Tensor | None = None,
        past_key_value: PastKeyValue | None = None,
        use_cache: bool = False,
    ) -> tuple[t.Tensor, PastKeyValue | None]:
        raise NotImplementedError()


tests.test_gemma_attention_shapes_and_cache(GemmaAttention)


## Exercise 6 - Decoder layer, full model, and HF reference parity

Assemble the RMSNorm, attention, and MLP pieces into a tiny Gemma causal LM. The final test loads weights from an independent `transformers.GemmaForCausalLM` with the same tiny config, then compares logits and cache behavior.


In [ ]:
class GemmaDecoderLayer(nn.Module):
    def __init__(self, config: GemmaConfig, layer_idx: int):
        super().__init__()
        self.self_attn = GemmaAttention(config, layer_idx=layer_idx)
        self.mlp = GemmaMLP(config)
        self.input_layernorm = GemmaRMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.post_attention_layernorm = GemmaRMSNorm(config.hidden_size, eps=config.rms_norm_eps)


    def forward(
        self,
        hidden_states: t.Tensor,
        *,
        position_ids: t.Tensor,
        cos: t.Tensor,
        sin: t.Tensor,
        attention_mask: t.Tensor | None = None,
        past_key_value: PastKeyValue | None = None,
        use_cache: bool = False,
    ) -> tuple[t.Tensor, PastKeyValue | None]:
        raise NotImplementedError()


tests.test_gemma_decoder_layer_shapes_and_cache(GemmaDecoderLayer)


## Exercise 7 - End-to-end tiny Gemma

Finish the model wrapper, then run the shape, cache-parity, and independent Hugging Face reference-parity checks. Passing this cell means the notebook no longer relies on the checked-in solution decoder for the learner's implementation.


<details><summary>Expected output</summary>

```text
All tests in `test_tiny_gemma_forward_shape` passed!
All tests in `test_tiny_gemma_cache_parity` passed!
All tests in `test_tiny_gemma_matches_reference_decoder` passed!
```

</details>

<details><summary>Help - what does HF parity prove, and what does it not prove?</summary>

It proves architecture parity against the public `transformers` Gemma reference
on a deterministic tiny config with copied random weights. It does not prove that
a gated pretrained Gemma checkpoint has been loaded or that any real-model
interpretability claim is true.

</details>


In [ ]:
class GemmaModel(nn.Module):
    def __init__(self, config: GemmaConfig):
        super().__init__()
        self.config = config
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)
        self.layers = nn.ModuleList(
            [GemmaDecoderLayer(config, layer_idx=i) for i in range(config.num_hidden_layers)]
        )
        self.norm = GemmaRMSNorm(config.hidden_size, eps=config.rms_norm_eps)

    def forward(
        self,
        input_ids: t.Tensor,
        *,
        attention_mask: t.Tensor | None = None,
        past_key_values: tuple[PastKeyValue, ...] | None = None,
        use_cache: bool = False,
    ) -> tuple[t.Tensor, tuple[PastKeyValue, ...] | None]:

        raise NotImplementedError()


class GemmaForCausalLM(nn.Module):
    def __init__(self, config: GemmaConfig):
        super().__init__()
        self.config = config
        self.model = GemmaModel(config)
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        if config.tie_word_embeddings:
            self.lm_head.weight = self.model.embed_tokens.weight

    def forward(
        self,
        input_ids: t.Tensor,
        *,
        attention_mask: t.Tensor | None = None,
        past_key_values: tuple[PastKeyValue, ...] | None = None,
        use_cache: bool = False,
    ) -> GemmaCausalLMOutput:
        raise NotImplementedError()


def make_tiny_gemma_config(sliding_window: int | None = None) -> GemmaConfig:
    return GemmaConfig(
        vocab_size=31,
        hidden_size=16,
        intermediate_size=32,
        num_hidden_layers=2,
        num_attention_heads=4,
        num_key_value_heads=2,
        max_position_embeddings=64,
        sliding_window=sliding_window,
    )


def make_tiny_gemma(seed: int = 0, sliding_window: int | None = None) -> GemmaForCausalLM:
    t.manual_seed(seed)
    return GemmaForCausalLM(make_tiny_gemma_config(sliding_window=sliding_window))


tests.test_tiny_gemma_forward_shape(make_tiny_gemma)
tests.test_tiny_gemma_cache_parity(make_tiny_gemma)
tests.test_tiny_gemma_matches_reference_decoder(GemmaForCausalLM)


## Local memory budget

The full Gemma 1B path is validated separately by the CUDA verification report. This notebook still ends with the same 24GB feasibility estimate used throughout the extension.


In [ ]:
budget = estimate_inference_memory(
    num_parameters=1_000_000_000,
    dtype="bfloat16",
    batch_size=1,
    context_length=2048,
    hidden_size=2048,
    num_layers=18,
    num_key_value_heads=8,
    head_dim=256,
    overhead_gb=1.5,
)
utils.print_report("Estimated Gemma 1B inference memory", budget.as_dict())
assert budget.fits(24.0), "The 1B bf16 inference estimate should fit the 24GB course target."


## Full Verification Contract

The smoke tests check the local exercise implementation. This final cell checks the committed CUDA verification report for the section-scale run and exposes the same `run_gpu_test` / `run_full_experiment` surface used by the release gate.


## Signature result

The committed CUDA report for this notebook was regenerated on an RTX 5090 Laptop
GPU with `torch 2.12.1+cu132`.

| Check | Observed result |
| --- | ---: |
| HF tiny-reference max absolute logit diff | `6.8769e-05` |
| HF tiny-reference MSE | `5.3475e-10` |
| HF tiny-reference KL divergence | `1.4670e-08` |
| HF tiny-reference top-k agreement | `1.0000` |
| Local cache max absolute diff | `1.9073e-06` |
| HF-reference cache max absolute diff | `4.4703e-08` |
| Total memory estimate | `3.6595 GB` |
| Measured peak VRAM | `0.0313 GB` |

```text
prompt:       [1, 5, 8, 13]
local:        [1, 5, 8, 13, 13, 13, 13, 13]
HF reference: [1, 5, 8, 13, 13, 13, 13, 13]
```

<details><summary>Interpreting this result</summary>

The repeated token is not the interesting part. The interesting part is that the
from-scratch model, cached generation path, and independent Hugging Face
reference all agree under the same architecture conventions. This is the
contract later real-checkpoint notebooks build on.

</details>

<details><summary>Limitations</summary>

This is a GT-0 architecture contract. It does not load gated pretrained Gemma
weights, evaluate natural-language quality, or validate a pretrained Gemma
mechanistic-interpretability claim.

</details>


In [ ]:
def _load_committed_gpu_report() -> dict:
    import json

    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    assert gpu["cuda_available"]
    assert gpu["within_vram_budget"]
    return gpu


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


gpu = _load_committed_gpu_report()
{key: gpu[key] for key in [
    "device",
    "preflight_passed",
    "peak_vram_gb",
] if key in gpu}
